In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from scipy.optimize import minimize
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
    SGDRegressor
)

In [22]:
df=pd.read_csv(r"marketing_spend_daily_2023_2025.csv")
df.head()

,Date,Region,TV_Spend,SocialMedia_Spend,GoogleAds_Spend,Influencer_Spend,Email_Spend,Total_Sales
0,2023-01-01,North,65795,43665,46784,30274,10774,617171
1,2023-01-01,South,50860,19277,34605,32399,6766,438377
2,2023-01-01,East,126820,52239,37463,36481,6633,788454
3,2023-01-01,West,104886,57664,28419,5944,3821,629799
4,2023-01-01,Central,56265,58008,67952,33457,3185,696821


In [7]:
features = ['TV_Spend','SocialMedia_Spend','GoogleAds_Spend','Influencer_Spend','Email_Spend']
X = df[features]
y = df['Total_Sales']

In [8]:
# imputer = SimpleImputer(strategy='median')
# df[num_cols] = imputer.fit_transform(df[num_cols])

In [9]:
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['dayofweek'] = df['Date'].dt.dayofweek
    df['month'] = df['Date'].dt.month

In [10]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, 'scaler1.joblib')

['scaler1.joblib']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=50)

print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")

Training samples: 4052, Test samples: 1013


In [12]:
# Train Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)
results = {}

In [13]:
preds = model.predict(X_test)
rmse = mean_squared_error(y_test, preds)
r2 = r2_score(y_test, preds)

In [14]:
preds

array([756036.82381073, 533707.59331808, 728572.08217044, ...,
       671635.16754725, 634424.72523072, 700589.62152525])

In [15]:
rmse

357716879.3605345

In [16]:
r2

0.9747069701061328

In [17]:
models = {
    "LinearRegression": LinearRegression(),
    "Ridge (sag)": Ridge(solver='sag'),
    "Lasso (cd)": Lasso(alpha=0.05),
    "ElasticNet (saga)": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "SGDRegressor": SGDRegressor(max_iter=1000, tol=1e-2)
}
results = {}

NameError: name 'Ridge' is not defined

In [ ]:
for name, model in models.items():
    print(f"\n🔹 Training {name}")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    rmse = mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    results[name] = {'model': model, 'rmse': rmse, 'r2': r2}
    print(f"RMSE: {rmse:.4f}, R²: {r2:.4f}")

In [ ]:
best_model_name = min(results, key=lambda k: results[k]['rmse'])
best_model = results[best_model_name]['model']
print(f"\n✅ Best model: {best_model_name}")
joblib.dump(best_model, 'best_ads_model1.joblib')

In [ ]:
preds = best_model.predict(X_test)
rmse = mean_squared_error(y_test, preds)
r2 = r2_score(y_test, preds)

In [ ]:
print(f"Test RMSE: {rmse:.4f}, R²: {r2:.4f}")

In [ ]:
plt.figure(figsize=(7,6))
plt.scatter(y_test, preds, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title('Predicted vs Actual')
plt.show()

plt.figure(figsize=(7,4))
residuals = y_test - preds
plt.hist(residuals, bins=30)
plt.title('Residual Distribution')
plt.show()

In [ ]:
# Reload the model and scaler
model = joblib.load('best_ads_model.joblib')
scaler = joblib.load('scaler.joblib')

channel_names = X.columns.tolist()
num_channels = len(channel_names)
TOTAL_BUDGET = 100000  # adjust as needed

def predict_sales(budgets):
    x = np.array(budgets).reshape(1, -1)
    x_scaled = scaler.transform(x)
    return model.predict(x_scaled)[0]

# Starting point: equal allocation
x0 = np.array([TOTAL_BUDGET / num_channels] * num_channels)

# Balanced constraints: 10–50% of total budget per channel
bounds = [(0.1 * TOTAL_BUDGET, 0.5 * TOTAL_BUDGET)] * num_channels

# Total budget must equal 100%
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - TOTAL_BUDGET},)

print("Running balanced optimization...")
res = minimize(lambda x: -predict_sales(x), x0=x0, bounds=bounds, constraints=constraints)

if res.success:
    optimal_budgets = res.x
    print("\n✅ Optimal balanced allocation:")
    for name, val in zip(channel_names, optimal_budgets):
        print(f"{name}: ₹{val:,.2f}")
    print(f"Predicted Sales: ₹{predict_sales(optimal_budgets):,.2f}")
else:
    print("❌ Optimization failed:", res.message)